In [1]:
import duckdb

query = """
WITH projected AS (
    -- 1. Project to metric
    -- 2. ST_MakeValid fixes any broken, self-intersecting polygons that cause C++ crashes
    SELECT 
        ST_MakeValid(ST_Transform(geometry, 'EPSG:4326', 'EPSG:28992')) as geom
    FROM read_parquet('./area_15.parquet')
    WHERE geometry IS NOT NULL
    LIMIT 50000 -- TEST RUN: Let's prove the math works on 50k rows first
),
buckets AS (
    -- Back to the 10m grid to keep the math localized
    SELECT 
        FLOOR(ST_X(ST_Centroid(geom)) / 10) AS grid_x,
        FLOOR(ST_Y(ST_Centroid(geom)) / 10) AS grid_y,
        geom
    FROM projected
),
micro_unions AS (
    SELECT ST_Union_Agg(geom) AS chunk_geom
    FROM buckets
    GROUP BY grid_x, grid_y
)
SELECT ST_Area(ST_Union_Agg(chunk_geom)) AS total_area_sqm
FROM micro_unions;
"""

con = duckdb.connect()

# Set memory limits just in case
con.execute("PRAGMA memory_limit='8GB';")
con.execute("PRAGMA temp_directory='./duckdb_temp';") 

con.execute("INSTALL spatial; LOAD spatial;")

print("Starting spatial calculation. If this takes longer than a minute, GEOS is working hard...")
result = con.sql(query)
result.show()

con.close()

Starting spatial calculation. If this takes longer than a minute, GEOS is working hard...
┌───────────────────┐
│  total_area_sqm   │
│      double       │
├───────────────────┤
│ 22251.94140938333 │
└───────────────────┘



In [4]:
import duckdb

con = duckdb.connect()

# Simple aggregation: Group by area, find min/max time
query = """
    SELECT 
        area, 
        MIN(time) AS start_datetime, 
        MAX(time) AS end_datetime,
        COUNT(*) AS row_count
    FROM read_parquet('./area_15.parquet')
    GROUP BY area
    ORDER BY start_datetime ASC
"""

print("Extracting time bounds per area...")
con.sql(query).show()

con.close()

Extracting time bounds per area...
┌─────────────────────┬──────────────────────────┬──────────────────────────┬───────────┐
│        area         │      start_datetime      │       end_datetime       │ row_count │
│       varchar       │ timestamp with time zone │ timestamp with time zone │   int64   │
├─────────────────────┼──────────────────────────┼──────────────────────────┼───────────┤
│ A_28DE699F_68EBF97D │ 2025-10-14 02:00:00+02   │ 2025-10-16 23:58:58+02   │   2708277 │
└─────────────────────┴──────────────────────────┴──────────────────────────┴───────────┘

